In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, accuracy_score
import os

# --- 1. Define File Paths and Utterance Lists (Unchanged) ---
base_path = '/kaggle/input/emguka-trial-corpus/EMG-UKA-Trial-Corpus/'
speaker_id = '002'
session_id = '001'

train_utterance_ids = [
    '0133', '0118', '0130', '0106', '0142', '0114', '0147', '0119', '0135', '0107',
    '0108', '0148', '0103', '0144', '0140', '0104', '0120', '0134', '0132', '0102',
    '0115', '0125', '0139', '0121', '0113', '0145', '0138', '0131', '0129', '0105',
    '0141', '0126', '0101', '0100', '0123', '0116', '0112', '0110', '0109', '0127'
]
test_utterance_ids = [
    '0122', '0143', '0146', '0149', '0117', '0137', '0124', '0111', '0128', '0136'
]

# --- 2. Data Loading Functions (Unchanged) ---
def read_emg_data(file_path, num_channels=7):
    try: return np.fromfile(file_path, dtype='<i2').reshape(-1, num_channels).T
    except FileNotFoundError: return None

def read_offset(file_path):
    try:
        with open(file_path, 'r') as f:
            start, end = map(int, f.readlines()[1].strip().split())
            return start, end
    except FileNotFoundError: return None, None

def read_alignments(file_path):
    try:
        alignments = []
        with open(file_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 3 and parts[2] != 'SIL':
                    alignments.append({'start': int(parts[0]), 'end': int(parts[1]), 'phone': parts[2]})
        return alignments
    except FileNotFoundError: return None

# --- 3. Feature Extraction with Stability Fix ---
def extract_features(frame):
    """
    Extracts a set of time-domain features from a single EMG frame.
    """
    epsilon = 1e-10
    features = []
    for i in range(frame.shape[1]):
        channel_data = frame[:, i].astype(np.float64)
        mav = np.mean(np.abs(channel_data))
        # CRITICAL FIX IS HERE
        rms = np.sqrt(np.mean(channel_data**2) + epsilon)
        wl = np.sum(np.abs(np.diff(channel_data)))
        zc = ((channel_data[:-1] * channel_data[1:]) < 0).sum()
        features.extend([mav, rms, wl, zc])
    return np.array(features)

# --- 4. Data Processing with Your Midpoint Alignment Logic ---
def process_utterances_with_features(utterance_ids, base_path, speaker_id, session_id):
    all_X_features, all_y = [], []
    SAMPLING_RATE_HZ, FRAME_SHIFT_MS, FRAME_SIZE_MS = 600, 10, 27
    frame_shift_samples = int(SAMPLING_RATE_HZ * (FRAME_SHIFT_MS / 1000.0))
    frame_size_samples = int(SAMPLING_RATE_HZ * (FRAME_SIZE_MS / 1000.0))

    for uid in utterance_ids:
        emg_path = os.path.join(base_path, 'emg', speaker_id, session_id, f'e07_{speaker_id}{session_id}{uid}.adc')
        offset_path = os.path.join(base_path, 'offset', speaker_id, session_id, f'offset_{speaker_id}{session_id}{uid}.txt')
        align_path = os.path.join(base_path, 'Alignments', speaker_id, session_id, f'phones_{speaker_id}{session_id}{uid}.txt')

        raw_emg = read_emg_data(emg_path)
        start_sample, end_sample = read_offset(offset_path)
        alignments = read_alignments(align_path)

        if raw_emg is None or start_sample is None or not alignments:
            print(f"Warning: Skipping utterance {uid} due to missing data or alignments.")
            continue

        emg_signal = raw_emg[:, start_sample:end_sample]
        max_frame = max(a['end'] for a in alignments)
        # Your logic correctly includes 'SIL' now as a potential class
        frame_labels = ['SIL'] * (max_frame + 1)
        for align in alignments:
            for i in range(align['start'], align['end'] + 1):
                frame_labels[i] = align['phone']

        num_frames = (emg_signal.shape[1] - frame_size_samples) // frame_shift_samples + 1
        for i in range(num_frames):
            start = i * frame_shift_samples
            end = start + frame_size_samples
            if end > emg_signal.shape[1]:
                break

            # Your improved midpoint logic
            midpoint = start + frame_size_samples // 2
            frame_idx = midpoint // frame_shift_samples

            if frame_idx < len(frame_labels):
                phone = frame_labels[frame_idx]
                frame_data = emg_signal[:, start:end].T
                features = extract_features(frame_data)
                all_X_features.append(features)
                all_y.append(phone)

    return np.array(all_X_features), np.array(all_y)

# --- 5. Training Phase ---
print("--- Phase 1: Loading and Extracting Features from Training Data ---")
X_train_features, y_train_raw = process_utterances_with_features(train_utterance_ids, base_path, speaker_id, session_id)

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train_raw)
y_train_categorical = to_categorical(y_train_encoded)
num_classes = y_train_categorical.shape[1]
print(f"Loaded and extracted features from {X_train_features.shape[0]} frames.")
print(f"Number of features per frame: {X_train_features.shape[1]}")
print(f"Found {num_classes} unique phoneme classes (including SIL).")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_features)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_scaled, y_train_categorical, test_size=0.2, random_state=42, stratify=y_train_categorical
)

# --- 6. Build and Train Model ---
print("\n--- Building a Dense Neural Network for feature classification ---")
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    BatchNormalization(), Dropout(0.5),
    Dense(256, activation='relu'),
    BatchNormalization(), Dropout(0.5),
    Dense(128, activation='relu'),
    BatchNormalization(), Dropout(0.5),
    Dense(num_classes, activation='softmax')
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=7, min_lr=0.00001)

print("\n--- Starting model training ---")
history = model.fit(
    X_train, y_train,
    epochs=150, batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)
print("--- Model Training Complete ---")

# --- 7. Testing Phase ---
print("\n--- Phase 2: Evaluating the Model on Test Data ---")
X_test_features, y_test_raw = process_utterances_with_features(test_utterance_ids, base_path, speaker_id, session_id)

if X_test_features.shape[0] > 0:
    print(f"Loaded and extracted features from {X_test_features.shape[0]} test frames.")
    X_test_scaled = scaler.transform(X_test_features)
    y_test_encoded = np.array([label_encoder.transform([label])[0] if label in label_encoder.classes_ else -1 for label in y_test_raw])

    valid_indices = y_test_encoded != -1
    X_test = X_test_scaled[valid_indices]
    y_test_true_labels = label_encoder.inverse_transform(y_test_encoded[valid_indices])

    predictions = np.argmax(model.predict(X_test), axis=1)
    y_pred_labels = label_encoder.inverse_transform(predictions)

    print("\n--- TEST RESULTS ---")
    accuracy = accuracy_score(y_test_true_labels, y_pred_labels)
    print(f"Overall Accuracy on Test Set: {accuracy * 100:.2f}%\n")
    print("Classification Report:")
    print(classification_report(y_test_true_labels, y_pred_labels, zero_division=0))
else:
    print("No valid test data was loaded.")